NAME: JEGATHEESWARI R

REG NO: 212223230092

In [2]:
import torch
import torch.nn as nn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
%matplotlib inline

In [3]:
df = pd.read_csv('incomee.csv')

In [4]:
print(len(df))
df.head(6)

30000


,age,sex,education,education-num,marital-status,workclass,occupation,hours-per-week,income,label
0,27,Male,HS-grad,9,Never-married,Private,Craft-repair,40,<=50K,0
1,47,Male,Masters,14,Married,Local-gov,Exec-managerial,50,>50K,1
2,59,Male,HS-grad,9,Divorced,Self-emp,Prof-specialty,20,<=50K,0
3,38,Female,Prof-school,15,Never-married,Federal-gov,Prof-specialty,57,>50K,1
4,64,Female,11th,7,Widowed,Private,Farming-fishing,40,<=50K,0
5,53,Male,Masters,14,Married,Private,Prof-specialty,40,>50K,1


In [5]:
df['label'].value_counts()

label
0    21700
1     8300
Name: count, dtype: int64

In [6]:
df.columns

Index(['age', 'sex', 'education', 'education-num', 'marital-status',
       'workclass', 'occupation', 'hours-per-week', 'income', 'label'],
      dtype='str')

In [7]:
# CODE HERE


cat_cols = ['workclass', 'education', 'marital-status', 'occupation', 'sex']

cont_cols = ['age', 'hours-per-week']

y_col = ['label']




# RUN THIS CODE TO COMPARE RESULTS:
print(f'cat_cols  has {len(cat_cols)} columns')
print(f'cont_cols has {len(cont_cols)} columns')
print(f'y_col     has {len(y_col)} column')

cat_cols  has 5 columns
cont_cols has 2 columns
y_col     has 1 column


In [8]:
# DON'T WRITE HERE

### 2. Convert categorical columns to category dtypes

In [9]:
# CODE HERE


for col in cat_cols:
    df[col] = df[col].astype('category')




In [10]:
# DON'T WRITE HERE

### Optional: Shuffle the dataset
The <strong>income.csv</strong> dataset is already shuffled. However, if you would like to try different configurations after completing the exercises, this is where you would want to shuffle the entire set.

In [11]:
# THIS CELL IS OPTIONAL
df = shuffle(df, random_state=101)
df.reset_index(drop=True, inplace=True)
df.head()

,age,sex,education,education-num,marital-status,workclass,occupation,hours-per-week,income,label
0,23,Female,HS-grad,9,Never-married,Private,Other-service,50,<=50K,0
1,37,Female,Prof-school,15,Married,State-gov,Prof-specialty,39,>50K,1
2,34,Male,Some-college,10,Divorced,Private,Adm-clerical,40,<=50K,0
3,31,Male,HS-grad,9,Married,Private,Craft-repair,40,>50K,1
4,20,Female,Some-college,10,Never-married,Private,Sales,25,<=50K,0


### 3. Set the embedding sizes
Create a variable "cat_szs" to hold the number of categories in each variable.<br>
Then create a variable "emb_szs" to hold the list of (category size, embedding size) tuples.

In [12]:
# CODE HERE

cat_szs = [len(df[col].cat.categories) for col in cat_cols]

emb_szs = [(size, min(50, (size + 1) // 2)) for size in cat_szs]


In [13]:
# DON'T WRITE HERE

### 4. Create an array of categorical values
Create a NumPy array called "cats" that contains a stack of each categorical column <tt>.cat.codes.values</tt><br>
Note: your output may contain different values. Ours came after performing the shuffle step shown above.

In [14]:
# CODE HERE

cats = np.stack([df[col].cat.codes.values for col in cat_cols], axis=1)

cats = cats.astype(np.int64)

# RUN THIS CODE TO COMPARE RESULTS
cats[:5]

array([[ 2, 10,  3,  6,  0],
       [ 4, 12,  1,  7,  0],
       [ 2, 13,  0,  0,  1],
       [ 2, 10,  1,  1,  1],
       [ 2, 13,  3,  9,  0]])

In [15]:
# DON'T WRITE HERE

### 5. Convert "cats" to a tensor
Convert the "cats" NumPy array to a tensor of dtype <tt>int64</tt>

In [16]:
# CODE HERE
cats = torch.tensor(cats, dtype=torch.int64)

In [17]:
# DON'T WRITE HERE

### 6. Create an array of continuous values
Create a NumPy array called "conts" that contains a stack of each continuous column.<br>
Note: your output may contain different values. Ours came after performing the shuffle step shown above.

In [18]:
# CODE HERE
conts = np.stack([df[col].values for col in cont_cols], axis=1)

conts = conts.astype(np.float32)


# RUN THIS CODE TO COMPARE RESULTS
conts[:5]

array([[23., 50.],
       [37., 39.],
       [34., 40.],
       [31., 40.],
       [20., 25.]], dtype=float32)

In [19]:
# DON'T WRITE HERE

### 7. Convert "conts" to a tensor
Convert the "conts" NumPy array to a tensor of dtype <tt>float32</tt>

In [20]:
# CODE HERE
conts = torch.tensor(conts, dtype=torch.float)

# RUN THIS CODE TO COMPARE RESULTS
conts.dtype

torch.float32

In [21]:
# DON'T WRITE HERE

### 8. Create a label tensor
Create a tensor called "y" from the values in the label column. Be sure to flatten the tensor so that it can be passed into the CE Loss function.

In [24]:
label_col = ['label']

y = torch.tensor(df[label_col].values.flatten(), dtype=torch.long)

In [ ]:
# DON'T WRITE HERE

### 9. Create train and test sets from <tt>cats</tt>, <tt>conts</tt>, and <tt>y</tt>
We use the entire batch of 30,000 records, but a smaller batch size will save time during training.<br>
We used a test size of 5,000 records, but you can choose another fixed value or a percentage of the batch size.<br>
Make sure that your test records remain separate from your training records, without overlap.<br>
To make coding slices easier, we recommend assigning batch and test sizes to simple variables like "b" and "t".

In [25]:
# CODE HERE
batch_size = 30000 # suggested batch size
test_size = 5000  # suggested test size
cat_train = cats[:batch_size-test_size]
cat_test = cats[batch_size-test_size:batch_size]

con_train = conts[:batch_size-test_size]
con_test = conts[batch_size-test_size:batch_size]

y_train = y[:batch_size-test_size]
y_test = y[batch_size-test_size:batch_size]


In [ ]:
# DON'T WRITE HERE

### Define the model class
Run the cell below to define the TabularModel model class we've used before.

In [26]:
class TabularModel(nn.Module):

    def __init__(self, emb_szs, n_cont, out_sz, layers, p=0.5):
        # Call the parent __init__
        super().__init__()
        
        # Set up the embedding, dropout, and batch normalization layer attributes
        self.embeds = nn.ModuleList([nn.Embedding(ni, nf) for ni,nf in emb_szs])
        self.emb_drop = nn.Dropout(p)
        self.bn_cont = nn.BatchNorm1d(n_cont)
        
        # Assign a variable to hold a list of layers
        layerlist = []
        
        # Assign a variable to store the number of embedding and continuous layers
        n_emb = sum((nf for ni,nf in emb_szs))
        n_in = n_emb + n_cont
        
        # Iterate through the passed-in "layers" parameter (ie, [200,100]) to build a list of layers
        for i in layers:
            layerlist.append(nn.Linear(n_in,i)) 
            layerlist.append(nn.ReLU(inplace=True))
            layerlist.append(nn.BatchNorm1d(i))
            layerlist.append(nn.Dropout(p))
            n_in = i
        layerlist.append(nn.Linear(layers[-1],out_sz))
        
        # Convert the list of layers into an attribute
        self.layers = nn.Sequential(*layerlist)
    
    def forward(self, x_cat, x_cont):
        # Extract embedding values from the incoming categorical data
        embeddings = []
        for i,e in enumerate(self.embeds):
            embeddings.append(e(x_cat[:,i]))
        x = torch.cat(embeddings, 1)
        # Perform an initial dropout on the embeddings
        x = self.emb_drop(x)
        
        # Normalize the incoming continuous data
        x_cont = self.bn_cont(x_cont)
        x = torch.cat([x, x_cont], 1)
        
        # Set up model layers
        x = self.layers(x)
        return x

### 10. Set the random seed
To obtain results that can be recreated, set a torch manual_seed (we used 33).

In [27]:
# CODE HERE
torch.manual_seed(32)

In [ ]:
# DON'T WRITE HERE

### 11. Create a TabularModel instance
Create an instance called "model" with one hidden layer containing 50 neurons and a dropout layer p-value of 0.4

In [28]:
model = TabularModel(
    emb_szs,
    n_cont=2,
    out_sz=2,
    layers=[50, 30],
    p=0.4
)

model

TabularModel(
  (embeds): ModuleList(
    (0): Embedding(5, 3)
    (1): Embedding(14, 7)
    (2): Embedding(6, 3)
    (3): Embedding(12, 6)
    (4): Embedding(2, 1)
  )
  (emb_drop): Dropout(p=0.4, inplace=False)
  (bn_cont): BatchNorm1d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layers): Sequential(
    (0): Linear(in_features=22, out_features=50, bias=True)
    (1): ReLU(inplace=True)
    (2): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=50, out_features=30, bias=True)
    (5): ReLU(inplace=True)
    (6): BatchNorm1d(30, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Dropout(p=0.4, inplace=False)
    (8): Linear(in_features=30, out_features=2, bias=True)
  )
)

In [ ]:
# DON'T WRITE HERE

### 12. Define the loss and optimization functions
Create a loss function called "criterion" using CrossEntropyLoss<br>
Create an optimization function called "optimizer" using Adam, with a learning rate of 0.001

In [29]:
# CODE HERE

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# DON'T WRITE HERE

### Train the model
Run the cell below to train the model through 300 epochs. Remember, results may vary!<br>
After completing the exercises, feel free to come back to this section and experiment with different parameters.

In [30]:
import time
start_time = time.time()

epochs = 300
losses = []

for i in range(epochs):
    i+=1
    y_pred = model(cat_train, con_train)
    loss = criterion(y_pred, y_train)
    losses.append(loss)
    
    # a neat trick to save screen space:
    if i%25 == 1:
        print(f'epoch: {i:3}  loss: {loss.item():10.8f}')

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f'epoch: {i:3}  loss: {loss.item():10.8f}') # print the last line
print(f'\nDuration: {time.time() - start_time:.0f} seconds') # print the time elapsed

epoch:   1  loss: 0.87438530
epoch:  26  loss: 0.59141982
epoch:  51  loss: 0.52321154
epoch:  76  loss: 0.48152095
epoch: 101  loss: 0.45371357
epoch: 126  loss: 0.43494663
epoch: 151  loss: 0.40927574
epoch: 176  loss: 0.38594863
epoch: 201  loss: 0.36968270
epoch: 226  loss: 0.35464105
epoch: 251  loss: 0.34221676
epoch: 276  loss: 0.33527961
epoch: 300  loss: 0.32610133

Duration: 32 seconds


### 13. Plot the Cross Entropy Loss against epochs
Results may vary. The shape of the plot is what matters.

In [31]:
plt.plot(range(epochs), [loss.detach().numpy() for loss in losses])
plt.ylabel('CE Loss')
plt.xlabel('epoch')

Text(0.5, 0, 'epoch')

In [ ]:
# DON'T WRITE HERE

### 14. Evaluate the test set
With torch set to <tt>no_grad</tt>, pass <tt>cat_test</tt> and <tt>con_test</tt> through the trained model. Create a validation set called "y_val". Compare the output to <tt>y_test</tt> using the loss function defined above. Results may vary.

In [32]:
# CODE HERE

with torch.no_grad():
    y_val = model(cat_test, con_test)
    loss = criterion(y_val, y_test)


# RUN THIS CODE TO COMPARE RESULTS
print(f'CE Loss: {loss:.8f}')

CE Loss: 0.34583122


In [ ]:
# TO EVALUATE THE TEST SET

### 15. Calculate the overall percent accuracy
Using a for loop, compare the argmax values of the <tt>y_val</tt> validation set to the <tt>y_test</tt> set.

In [33]:
correct = 0

with torch.no_grad():
    y_val = model(cat_test, con_test)
    predicted = torch.max(y_val, 1)[1]
    correct += (predicted == y_test).sum()

print(f'{correct.item()}/{len(y_test)} = {correct.item()*100/len(y_test):7.3f}% correct')

4230/5000 =  84.600% correct


In [ ]:
# DON'T WRITE HERE

In [34]:
# WRITE YOUR CODE HERE:
df.iloc[-1]

age                      50
sex                    Male
education         Bachelors
education-num            13
marital-status      Widowed
workclass          Self-emp
occupation            Sales
hours-per-week           65
income                 >50K
label                     1
Name: 29999, dtype: object

In [ ]:
# DON'T WRITE HERE